> 📘 **Solucionario.** Esta versión contiene las soluciones de todos los ejercicios; está pensada para el equipo docente.

<a href="https://colab.research.google.com/github/ibonfilrivera/Curso_Python_FQ/blob/main/soluciones/Sesion_3_soluciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# **Curso introductorio de Python**
## **Facultad de Química, UNAM** · Departamento de Física y Química Teórica

**Elaboraron:** Iván Bonfil, Rafael Rodriguez y Roberto Rojas

---

# **Sesión 3: Bibliotecas científicas y visualización de datos**


En esta sesión usaremos bibliotecas especializadas de Python para hacer cálculos numéricos,
ajustar datos experimentales, generar gráficas de calidad editorial y explorar una base de datos
con casi 10 000 compuestos.

**Al terminar podrás:**
- Hacer regresiones y resolver ecuaciones con SciPy.
- Operar con vectores y matrices usando NumPy.
- Construir gráficas científicas con Matplotlib (ejes, unidades, leyendas y exportación).
- Leer, filtrar y resumir datos tabulares con Pandas.
- Representar moléculas y buscar subestructuras con RDKit.

## **¿Cómo usar este notebook?**

Este notebook es **interactivo**: cada ejercicio tiene una celda de código para tu respuesta y
una celda que la **verifica automáticamente**, como en los cursos de [Kaggle Learn](https://www.kaggle.com/learn).

1. Ejecuta la celda de **configuración** que está justo abajo (una sola vez, al abrir el notebook).
2. En cada ejercicio, sustituye los espacios `____` por tu código y ejecuta la celda.
3. Ejecuta la celda `ejN.verificar()`. Verás uno de estos mensajes:
   - ✅ **¡Correcto!** — puedes continuar.
   - ❌ **Incorrecto** — el mensaje te dice qué revisar.
   - ✏️ **Pendiente** — aún falta completar algo.
4. Si te atoras, quita el `#` de `ejN.pista()` para recibir una pista, o de `ejN.solucion()`
   para ver una solución. **Intenta resolverlo antes de ver la solución.**
5. Ejecuta `progreso()` en cualquier momento para ver tu avance en la sesión.

> 💡 El verificador lee las variables del notebook. Si reinicias el entorno de ejecución, vuelve
> a ejecutar la celda de configuración y las celdas anteriores al ejercicio.

In [ ]:
# ⚙️ Configuración: ejecuta esta celda antes de empezar
import os
import sys
import urllib.request

REPOSITORIO = "https://raw.githubusercontent.com/ibonfilrivera/Curso_Python_FQ/main"

if os.path.isdir("../verificador"):      # Copia local del repositorio
    sys.path.insert(0, "..")
else:                                     # Google Colab: descarga el verificador
    os.makedirs("verificador", exist_ok=True)
    for archivo in ["__init__.py", "nucleo.py", "sesion3.py"]:
        urllib.request.urlretrieve(f"{REPOSITORIO}/verificador/{archivo}",
                                   f"verificador/{archivo}")

from verificador.sesion3 import *

## **Importar bibliotecas**

Una **biblioteca** (o *librería*) es un conjunto de funciones que alguien más escribió y que
podemos reutilizar. Se cargan con `import`, y es costumbre darles un alias corto:

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# **math, SymPy y SciPy**

El módulo `math` incluye funciones y constantes matemáticas básicas.

In [ ]:
print(math.sin(math.pi))       # Seno (el resultado es ~0 por redondeo numérico)
print(math.cos(math.pi / 3))   # Coseno
print(math.sqrt(100))          # Raíz cuadrada
print(math.log(math.e))        # Logaritmo natural
print(math.log10(100))         # Logaritmo base 10
print(math.log2(8))            # Logaritmo base 2

**SymPy** hace cálculo **simbólico**: manipula expresiones algebraicas en lugar de números.

In [ ]:
import sympy as sp

x = sp.symbols("x")
expresion = x**2 * sp.exp(-x)

print("Derivada:", sp.diff(expresion, x))
print("Integral:", sp.integrate(expresion, x))

**SciPy** reúne métodos numéricos, estadísticos y de optimización para el trabajo científico.
Por ejemplo, con `scipy.stats` podemos analizar una curva de calibración de absorbancia contra
concentración:

In [ ]:
from scipy.stats import describe, linregress

concentraciones = [0.0, 2.0, 4.0, 6.0, 8.0, 10.0]      # mg/L
absorbancias = [0.012, 0.225, 0.451, 0.645, 0.852, 1.058]

print(describe(absorbancias))                          # Estadística descriptiva

ajuste = linregress(concentraciones, absorbancias)     # Regresión lineal
print(f"Pendiente: {ajuste.slope:.4f} L/mg")
print(f"Ordenada:  {ajuste.intercept:.4f}")
print(f"r²:        {ajuste.rvalue**2:.5f}")

Con `curve_fit` de `scipy.optimize` podemos ajustar cualquier función, por ejemplo un polinomio de segundo grado:

In [ ]:
from scipy.optimize import curve_fit

def cuadratica(x, a, b, c):
    return a * x**2 + b * x + c

parametros, covarianza = curve_fit(cuadratica, concentraciones, absorbancias)
a, b, c = parametros
print(f"A = {a:.2e}·c² + {b:.4f}·c + {c:.4f}")

`scipy.optimize` también resuelve ecuaciones numéricamente (`fsolve`) y busca mínimos de
funciones (`minimize`):

In [ ]:
from scipy.optimize import fsolve, minimize

def polinomio(x):
    return (x + 2)**2 - 4

raiz = fsolve(polinomio, x0=-3)       # Busca una raíz cerca de x = -3
print(f"Raíz: x = {raiz[0]:.4f}")

minimo = minimize(polinomio, x0=3)    # Busca un mínimo empezando en x = 3
print(f"Mínimo en x = {minimo.x[0]:.4f}, f(x) = {minimo.fun:.4f}")

### **Ejercicio 1: Orden de reacción**

Se midió la concentración de un reactivo A a lo largo del tiempo. Las ecuaciones integradas de
velocidad son:

| Orden | Ecuación | Se grafica contra $t$ |
| :-: | :-: | :-: |
| 0 | $[A] = -kt + [A]_0$ | $[A]$ |
| 1 | $\ln[A] = -kt + \ln[A]_0$ | $\ln[A]$ |
| 2 | $\frac{1}{[A]} = kt + \frac{1}{[A]_0}$ | $\frac{1}{[A]}$ |

Haz las tres regresiones lineales con `linregress`, compara sus $r^2$ y guarda:
- `mejor_orden`: el orden (0, 1 o 2) que mejor describe los datos.
- `k`: la constante de velocidad (positiva) para ese orden.

In [ ]:
tiempo = [0, 10, 20, 30, 40, 50]                      # min
conc = [1.000, 0.607, 0.368, 0.223, 0.135, 0.082]     # mol/L

In [ ]:
ln_conc = [math.log(c) for c in conc]
inv_conc = [1 / c for c in conc]

ajustes = {
    0: linregress(tiempo, conc),
    1: linregress(tiempo, ln_conc),
    2: linregress(tiempo, inv_conc),
}
for orden, ajuste in ajustes.items():
    print(f"Orden {orden}: r² = {ajuste.rvalue**2:.5f}")

mejor_orden = max(ajustes, key=lambda o: ajustes[o].rvalue**2)
k = -ajustes[1].slope
print(f"Mejor ajuste: orden {mejor_orden}, k = {k:.4f} min⁻¹")

In [ ]:
# Verifica tu respuesta
ej1.verificar()

# **NumPy**

Las listas de Python son flexibles, pero lentas para cálculos numéricos. NumPy ofrece los
**arreglos** (`np.array`), que permiten operar con todos los elementos a la vez y son mucho más
rápidos. Comparemos el tiempo para elevar al cuadrado un millón de números:

In [ ]:
lista = list(range(1_000_000))
arreglo = np.array(lista)

inicio = time.perf_counter()
cuadrados_lista = [valor**2 for valor in lista]
tiempo_lista = time.perf_counter() - inicio

inicio = time.perf_counter()
cuadrados_arreglo = arreglo**2          # Operación vectorizada: sin ciclo explícito
tiempo_arreglo = time.perf_counter() - inicio

print(f"Con lista:   {tiempo_lista * 1000:.1f} ms")
print(f"Con arreglo: {tiempo_arreglo * 1000:.1f} ms  ({tiempo_lista / tiempo_arreglo:.0f} veces más rápido)")

Los arreglos representan vectores y matrices, y permiten las operaciones del álgebra lineal:

In [ ]:
matriz_A = np.array([[1, 2],
                     [3, 4]])
matriz_B = np.array([[5, 6],
                     [7, 8]])

# Producto elemento a elemento (NO es el producto de matrices)
print("A * B =\n", matriz_A * matriz_B)

# Producto matricial (filas por columnas)
print("A @ B =\n", matriz_A @ matriz_B)

# Transpuesta
print("Aᵀ =\n", matriz_A.T)

Al igual que las listas, los arreglos se indexan; además podemos extraer filas o columnas
completas con **rebanadas** (*slicing*): `arr[fila, columna]`, donde `:` significa "todas".

In [ ]:
arr = np.arange(9).reshape(3, 3)    # Números del 0 al 8 acomodados en una matriz 3×3
print(arr, "forma:", arr.shape)

print("Primera fila:", arr[0])
print("Elemento (0, 1):", arr[0, 1])
print("Primera columna:", arr[:, 0])

arr[0, :] = [10, 20, 30]            # Reemplazamos la primera fila
print(arr)

El módulo `np.linalg` contiene las funciones más comunes de álgebra lineal:

In [ ]:
print("Determinante de A:", np.linalg.det(matriz_A))

valores_propios, vectores_propios = np.linalg.eig(matriz_A)
print("Valores propios:", valores_propios)
print("Vectores propios (columnas):\n", vectores_propios)

In [ ]:
# Resolver el sistema   2x +  y +  z = 10
#                        x -  y + 2z =  5
#                       3x + 2y -  z =  7
coeficientes = np.array([[2,  1,  1],
                         [1, -1,  2],
                         [3,  2, -1]])
resultados = np.array([10, 5, 7])

solucion = np.linalg.solve(coeficientes, resultados)
print("x, y, z =", solucion)

### **Ejercicio 2: Matriz de rotación**

Para rotar un vector en $\mathbb{R}^2$ un ángulo $\theta$ se multiplica por la matriz de rotación:

$$R(\theta) = \begin{pmatrix}
\cos\theta & -\sin\theta \\
\sin\theta & \cos\theta
\end{pmatrix}$$

Escribe la función `rotar_vector(vector, angulo_grados)` que devuelva el vector rotado.

In [ ]:
def rotar_vector(vector, angulo_grados):
    theta = np.radians(angulo_grados)
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    return R @ np.asarray(vector, dtype=float)

print(rotar_vector([0, 1], 90))

In [ ]:
# Verifica tu respuesta
ej2.verificar()

### **Ejercicio 3: Regla de Cramer**

Para un sistema de $2 \times 2$

$$\begin{aligned}
ax + by &= e \\
cx + dy &= f
\end{aligned}$$

la regla de Cramer da $x = \frac{\Delta_x}{\Delta}$ y $y = \frac{\Delta_y}{\Delta}$, con

$$\Delta = \begin{vmatrix} a & b \\ c & d \end{vmatrix}, \quad
\Delta_x = \begin{vmatrix} e & b \\ f & d \end{vmatrix}, \quad
\Delta_y = \begin{vmatrix} a & e \\ c & f \end{vmatrix}$$

Escribe `resolver_cramer_2x2(A, b)` que devuelva un arreglo `[x, y]`, y compara tu resultado
con `np.linalg.solve`.

In [ ]:
def resolver_cramer_2x2(A, b):
    A = np.array(A, dtype=float)
    delta = np.linalg.det(A)

    A_x = A.copy()
    A_x[:, 0] = b
    A_y = A.copy()
    A_y[:, 1] = b

    return np.array([np.linalg.det(A_x), np.linalg.det(A_y)]) / delta

A = np.array([[3, 2], [4, -1]])
b = np.array([12, 5])
print(resolver_cramer_2x2(A, b), np.linalg.solve(A, b))

In [ ]:
# Verifica tu respuesta
ej3.verificar()

# **Matplotlib**

Matplotlib es la biblioteca de visualización más usada en ciencia. Aunque tiene varias formas
de uso, recomendamos la interfaz **orientada a objetos**:

```python
fig, ax = plt.subplots()      # fig: la figura completa; ax: los ejes donde se dibuja
ax.plot(x, y)                 # Dibujar
ax.set_xlabel("...")          # Personalizar
plt.show()                    # Mostrar
```

Una gráfica científica de calidad siempre tiene: **ejes etiquetados con unidades**, una
**leyenda** si hay más de una serie, y un tamaño de letra legible.

📎 Referencias: [hojas de referencia rápida](https://matplotlib.org/cheatsheets/) y
[galería de ejemplos](https://matplotlib.org/stable/gallery/index.html).

### **Gráfica de líneas: decaimiento radiactivo**

Retomemos el carbono-14 de la sesión anterior, ahora con $m(t) = m_0 \left(\frac{1}{2}\right)^{t/t_{1/2}}$.
Con NumPy generamos 200 tiempos de una sola vez con `np.linspace`.

In [ ]:
t_vida_media = 5730                              # años
t = np.linspace(0, 5 * t_vida_media, 200)        # 200 tiempos entre 0 y 5 vidas medias
masa = 1.0 * 0.5**(t / t_vida_media)             # g

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t, masa, color="tab:blue", linewidth=2, label="¹⁴C")
ax.axhline(0.5, color="gray", linestyle="--", label="Mitad de la masa inicial")
ax.set_xlabel("Tiempo (años)")
ax.set_ylabel("Masa (g)")
ax.set_title("Decaimiento radiactivo del carbono-14")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### **Varias gráficas en una figura**

`plt.subplots(filas, columnas)` crea una cuadrícula de ejes. Veamos los datos cinéticos del
Ejercicio 1 con las tres transformaciones: la que se vea como una recta indica el orden de reacción.

In [ ]:
tiempo_arr = np.array(tiempo)
conc_arr = np.array(conc)

transformaciones = [
    (conc_arr, "[A] (mol/L)", "Orden 0"),
    (np.log(conc_arr), "ln [A]", "Orden 1"),
    (1 / conc_arr, "1/[A] (L/mol)", "Orden 2"),
]

fig, ejes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (y, etiqueta, titulo) in zip(ejes, transformaciones):
    ajuste_orden = linregress(tiempo_arr, y)
    ax.plot(tiempo_arr, y, "o", label="Datos")
    ax.plot(tiempo_arr, ajuste_orden.slope * tiempo_arr + ajuste_orden.intercept, "--",
            label=f"r² = {ajuste_orden.rvalue**2:.4f}")
    ax.set_xlabel("t (min)")
    ax.set_ylabel(etiqueta)
    ax.set_title(titulo)
    ax.legend()

fig.tight_layout()     # Evita que se encimen las etiquetas
plt.show()

### **Guardar una figura**

Para un reporte o una tesis, exporta con alta resolución (300 ppp) o en formato vectorial (PDF,
SVG). En Colab, el archivo aparece en el panel de archivos 📁 de la izquierda.

```python
fig.savefig("cinetica.png", dpi=300, bbox_inches="tight")
fig.savefig("cinetica.pdf", bbox_inches="tight")
```

### **Ejercicio 4: Curva de calibración**

Con los datos de `concentraciones` y `absorbancias` de la sección de SciPy:

1. Crea la figura `fig_calibracion` con los puntos experimentales (`ax.scatter`) y la recta del
   ajuste lineal (`ax.plot`).
2. Etiqueta ambos ejes con sus unidades y agrega una leyenda.
3. Una muestra problema tiene una absorbancia de 0.500. Calcula su concentración con la recta
   de calibración y guárdala en `conc_problema`.

In [ ]:
ajuste = linregress(concentraciones, absorbancias)
x = np.array(concentraciones)

fig_calibracion, ax = plt.subplots(figsize=(6, 4))
ax.scatter(concentraciones, absorbancias, color="tab:blue", label="Datos experimentales")
ax.plot(x, ajuste.slope * x + ajuste.intercept, color="tab:red",
        label=f"A = {ajuste.slope:.4f}·c + {ajuste.intercept:.4f} (r² = {ajuste.rvalue**2:.4f})")
ax.set_xlabel("Concentración (mg/L)")
ax.set_ylabel("Absorbancia")
ax.set_title("Curva de calibración")
ax.legend()
plt.show()

conc_problema = (0.500 - ajuste.intercept) / ajuste.slope
print(f"Concentración de la muestra problema: {conc_problema:.2f} mg/L")

In [ ]:
# Verifica tu respuesta
ej4.verificar()

# **Pandas**

Pandas es la biblioteca más usada para analizar datos tabulares: lee archivos (CSV, Excel,
etc.), filtra, agrupa y grafica. Su estructura principal es el **DataFrame**, una tabla con
filas y columnas con nombre, similar a una hoja de cálculo.

Usaremos **AqSolDB**, una base de datos curada con la solubilidad acuosa de 9 982 compuestos
([Sorkun *et al.*, *Scientific Data* **6**, 143 (2019)](https://doi.org/10.1038/s41597-019-0151-1)).
Algunas de sus columnas son:

| Columna | Significado |
| :--- | :--- |
| `Name`, `SMILES` | Nombre y estructura del compuesto |
| `Solubility` | log S, con S la solubilidad en mol/L |
| `MolWt` | Masa molar (g/mol) |
| `MolLogP` | Coeficiente de partición octanol/agua (log P) |
| `NumHDonors`, `NumHAcceptors` | Donadores y aceptores de puentes de hidrógeno |
| `TPSA` | Área superficial polar (Å²) |

In [ ]:
from pathlib import Path

RUTA_LOCAL = Path("../data/curated_solubility.csv")
URL_DATOS = "https://raw.githubusercontent.com/ibonfilrivera/Curso_Python_FQ/main/data/curated_solubility.csv"

# Usa la copia local si existe; si no (por ejemplo, en Colab), descarga el archivo
df = pd.read_csv(RUTA_LOCAL if RUTA_LOCAL.exists() else URL_DATOS)

print(f"La tabla tiene {df.shape[0]} filas y {df.shape[1]} columnas")
df.head()

Hagamos un análisis preliminar. `describe()` resume las columnas numéricas:

In [ ]:
df[["Solubility", "MolWt", "MolLogP", "NumHDonors", "NumHAcceptors"]].describe()

Para **filtrar** filas escribimos una condición entre corchetes. Varias condiciones se combinan
con `&` (y) o `|` (o), y cada una va entre paréntesis:

In [ ]:
# Compuestos con masa molar menor a 50 g/mol
df[df["MolWt"] < 50][["Name", "MolWt", "Solubility"]]

In [ ]:
# Los 5 compuestos más solubles
df.sort_values("Solubility", ascending=False)[["Name", "Solubility"]].head()

Pandas se integra con Matplotlib. Por ejemplo, un histograma de la solubilidad:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["Solubility"], bins=50, color="tab:green", edgecolor="white")
ax.set_xlabel("log S (S en mol/L)")
ax.set_ylabel("Número de compuestos")
ax.set_title("Distribución de la solubilidad acuosa en AqSolDB")
plt.show()

### **Ejercicio 5: Regla de los 5 de Lipinski**

En el diseño de fármacos, la regla de Lipinski estima si un compuesto podría ser activo por vía
oral. Un buen candidato cumple **todas** estas condiciones:

- Masa molar ≤ 500 g/mol.
- log P ≤ 5.
- Donadores de puentes de hidrógeno ≤ 5.
- Aceptores de puentes de hidrógeno ≤ 10.

Guarda en `lipinski` las filas de `df` que cumplen la regla y en `porcentaje_lipinski` el
porcentaje de compuestos que la cumplen.

In [ ]:
lipinski = df[(df["MolWt"] <= 500)
              & (df["MolLogP"] <= 5)
              & (df["NumHDonors"] <= 5)
              & (df["NumHAcceptors"] <= 10)]

porcentaje_lipinski = 100 * len(lipinski) / len(df)
print(f"{len(lipinski)} moléculas ({porcentaje_lipinski:.1f} %) cumplen la regla de Lipinski")

In [ ]:
# Verifica tu respuesta
ej5.verificar()

### **Ejercicio 6 (integrador): lipofilicidad y solubilidad**

¿Los compuestos más lipofílicos son menos solubles en agua?

1. Crea la figura `fig_logp` con un diagrama de dispersión de `MolLogP` (eje x) contra
   `Solubility` (eje y), con ambos ejes etiquetados.
2. Calcula el coeficiente de correlación de Pearson entre ambas columnas y guárdalo en `r_logp`.
3. Interpreta: ¿qué signo tiene la correlación y qué significa químicamente?

In [ ]:
r_logp = df["MolLogP"].corr(df["Solubility"])

fig_logp, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["MolLogP"], df["Solubility"], s=5, alpha=0.3)
ax.set_xlabel("logP (octanol/agua)")
ax.set_ylabel("log S (mol/L)")
ax.set_title(f"Solubilidad acuosa contra lipofilicidad (r = {r_logp:.2f})")
plt.show()

In [ ]:
# Verifica tu respuesta
ej6.verificar()

# **RDKit: química computacional**

RDKit es una biblioteca de quimioinformática: interpreta estructuras moleculares (por ejemplo,
en notación **SMILES**), las dibuja y calcula propiedades. Como no viene instalada en Colab, la
instalamos primero.

In [ ]:
try:
    import rdkit
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True)

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

RDLogger.DisableLog("rdApp.*")   # Oculta advertencias de estructuras problemáticas

In [ ]:
aspirina = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
aspirina

Convertimos todos los SMILES de la tabla en objetos de RDKit (tarda unos segundos):

In [ ]:
df["Mols"] = df["SMILES"].apply(Chem.MolFromSmiles)

no_validas = df["Mols"].isna().sum()
print(f"Moléculas que RDKit no pudo interpretar: {no_validas}")

Draw.MolsToGridImage(df["Mols"][:8].tolist(), molsPerRow=4, subImgSize=(200, 200),
                     legends=[nombre[:25] for nombre in df["Name"][:8]])

### **Ejercicio 7: Búsqueda de subestructuras**

Con un patrón **SMARTS** podemos buscar fragmentos dentro de las moléculas. El anillo bencénico
aromático se escribe `"c1ccccc1"`.

1. Completa la función `tiene_benceno(mol)`, que devuelva `True` si la molécula contiene un
   anillo bencénico (y `False` si `mol` es `None`).
2. Aplícala para crear la columna `df["tiene_benceno"]`.
3. Guarda en `n_benceno` cuántos compuestos contienen benceno.

In [ ]:
patron_benceno = Chem.MolFromSmarts("c1ccccc1")

def tiene_benceno(mol):
    if mol is None:          # SMILES que RDKit no pudo interpretar
        return False
    return mol.HasSubstructMatch(patron_benceno)

df["tiene_benceno"] = df["Mols"].apply(tiene_benceno)
n_benceno = int(df["tiene_benceno"].sum())
print(f"{n_benceno} de {len(df)} moléculas contienen un anillo bencénico")

In [ ]:
# Verifica tu respuesta
ej7.verificar()

## **Resumen de la sesión**

| Biblioteca | Para qué sirve | Funciones clave |
| :--- | :--- | :--- |
| `math` | Funciones matemáticas básicas | `math.log`, `math.sqrt`, `math.pi` |
| SymPy | Cálculo simbólico | `sp.symbols`, `sp.diff`, `sp.integrate` |
| SciPy | Estadística, ajustes y ecuaciones | `linregress`, `curve_fit`, `fsolve` |
| NumPy | Arreglos y álgebra lineal | `np.array`, `@`, `np.linalg.solve` |
| Matplotlib | Gráficas | `plt.subplots`, `ax.plot`, `ax.scatter`, `fig.savefig` |
| Pandas | Datos tabulares | `pd.read_csv`, `df.describe`, `df[condición]` |
| RDKit | Quimioinformática | `Chem.MolFromSmiles`, `HasSubstructMatch` |

**Para seguir aprendiendo:**
- [Kaggle Learn: Pandas](https://www.kaggle.com/learn/pandas) y [Data Visualization](https://www.kaggle.com/learn/data-visualization).
- [Tutorial de introducción de RDKit](https://www.rdkit.org/docs/GettingStartedInPython.html).

## **Tu progreso**

Ejecuta la siguiente celda para ver cuántos ejercicios resolviste en esta sesión.

In [ ]:
progreso()